[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sadriica/Curso_ANH/blob/main/modulo3_multicriterio/modulo3_multicriterio.ipynb)

Primera vez en Colab: ver la [guía](https://github.com/Sadriica/Curso_ANH/blob/main/guia_colab.md). Términos: [glosario](https://github.com/Sadriica/Curso_ANH/blob/main/glosario.md).

# Data & GIS para Energía
## Módulo 3: ¿Cómo tomar una decisión? (Multicriterio)

Para decidir dónde conviene un proyecto no basta una sola variable: pesan el recurso (viento),
el acceso (vías), la infraestructura y lo social. Los métodos multicriterio combinan varias
variables, cada una con un peso, en un único puntaje de idoneidad.

Contenido:

1. Punto de partida: los datos del Módulo 2, recreados aquí para no depender del otro notebook.
2. Normalización de criterios (llevarlos a una escala común).
3. Suma ponderada (el método más directo).
4. AHP (pesos a partir de comparaciones por pares).
5. TOPSIS (cercanía a la solución ideal).
6. Comparativa: mismos datos, distinto método.

Nivel básico, corre completo en Colab. Los archivos usados están también en `recursos/`.

## 0. Preparación

In [ ]:
!pip install -q geopandas "h3>=4.1" folium mapclassify

In [ ]:
import pandas as pd
import numpy as np
import h3
import folium
import matplotlib.pyplot as plt

## 1. Punto de partida (resultado del Módulo 2)

El resultado del Módulo 2 es la malla H3 con las variables (`malla_h3.parquet`): cada fila es una
celda (`h3_index`) con sus criterios. Ese es el formato de trabajo del proyecto.

En un caso real se carga directamente con
`malla = pd.read_parquet("recursos/malla_h3.parquet")`; aquí se reconstruye para que el notebook
sea independiente. Cada celda H3 es una alternativa y cada variable es un criterio.

In [ ]:
RES = 5
municipios = pd.DataFrame({
    "municipio": ["Uribia", "Riohacha", "Maicao", "Manaure",
                  "Valledupar", "Aguachica", "Santa Marta", "Barranquilla"],
    "lat": [11.71, 11.55, 11.38, 11.78, 10.46, 8.31, 11.24, 10.96],
    "lon": [-71.98, -72.91, -72.24, -72.44, -73.25, -73.63, -74.20, -74.80],
    "viento_ms":   [9.1, 7.3, 6.8, 8.7, 5.2, 4.1, 5.9, 6.2],   # criterio: mas es mejor
    "dist_via_km": [3.2, 0.5, 1.1, 4.8, 0.3, 0.9, 0.4, 0.2],   # criterio: menos es mejor
    "radiacion":   [6.1, 5.8, 5.9, 6.0, 5.2, 5.0, 5.4, 5.5],   # criterio: mas es mejor
})
# cada celda H3 es una alternativa
municipios["h3_index"] = [h3.latlng_to_cell(la, lo, RES) for la, lo in zip(municipios["lat"], municipios["lon"])]
municipios

Cada criterio tiene un sentido: *beneficio* (más es mejor) o *costo* (menos es mejor).

In [ ]:
criterios = {
    "viento_ms":   "beneficio",   # mas viento, mejor
    "dist_via_km": "costo",       # mas lejos de una via, peor
    "radiacion":   "beneficio",   # mas radiacion, mejor
}
list(criterios.items())

## 2. Normalización de criterios

Los criterios están en unidades distintas (m/s, km, kWh). Para combinarlos hay que llevarlos a
una escala común de 0 a 1, donde 1 es siempre lo mejor. En los criterios de costo la escala se
invierte: menos es mejor, y por tanto más cerca de 1. El método usado es min-max.

In [ ]:
def normalizar(col, sentido):
    v = municipios[col].astype(float)
    z = (v - v.min()) / (v.max() - v.min())     # 0 a 1
    return z if sentido == "beneficio" else 1 - z

norm = pd.DataFrame({"municipio": municipios["municipio"]})
for col, sentido in criterios.items():
    norm[col] = normalizar(col, sentido)
norm.round(2)

## 3. Suma ponderada

Es el método más directo: a cada criterio se le asigna un peso, los pesos suman 1, y el puntaje
de cada alternativa es la suma ponderada de sus valores normalizados.

In [ ]:
pesos = {"viento_ms": 0.5, "dist_via_km": 0.2, "radiacion": 0.3}   # deben sumar 1
assert abs(sum(pesos.values()) - 1) < 1e-9

municipios["puntaje_suma"] = sum(norm[c] * w for c, w in pesos.items())
municipios[["municipio", "puntaje_suma"]].sort_values("puntaje_suma", ascending=False).round(3)

## 4. AHP (Analytic Hierarchy Process)

En la suma ponderada los pesos se fijan directamente. AHP los deriva de comparaciones por pares:
para cada par de criterios se responde cuál es más importante y cuánto, en una escala de 1 a 9.
De esa matriz se obtienen los pesos y se puede medir qué tan consistentes fueron los juicios.

Matriz de comparación (fila vs columna). Aquí: viento un poco más importante que radiación, y
bastante más que la distancia a vías.

In [ ]:
crit = list(criterios.keys())   # ["viento_ms", "dist_via_km", "radiacion"]
# A[i][j]: cuanto mas importante es el criterio i que el j
A = np.array([
    [1,   4,   2  ],   # viento vs (viento, dist_via, radiacion)
    [1/4, 1,   1/2],   # dist_via
    [1/2, 2,   1  ],   # radiacion
], dtype=float)
pd.DataFrame(A, index=crit, columns=crit)

In [ ]:
# Pesos: normalizar cada columna y promediar por fila
col_norm = A / A.sum(axis=0)
pesos_ahp = col_norm.mean(axis=1)
pesos_ahp = dict(zip(crit, pesos_ahp))
{c: round(w, 3) for c, w in pesos_ahp.items()}

### Consistencia

AHP permite chequear si los juicios fueron coherentes (por ejemplo, no decir "A>B, B>C, pero C>A").
Se calcula la razón de consistencia (CR); como regla, CR < 0.1 es aceptable.

In [ ]:
w = np.array([pesos_ahp[c] for c in crit])
lamb_max = (A @ w / w).mean()
n = len(crit)
CI = (lamb_max - n) / (n - 1)                 # indice de consistencia
RI = {1:0, 2:0, 3:0.58, 4:0.90, 5:1.12}[n]    # indice aleatorio (tabla estandar)
CR = CI / RI if RI else 0
print(f"CR = {CR:.3f}  ->  {'aceptable' if CR < 0.1 else 'revisar juicios'}")

In [ ]:
municipios["puntaje_ahp"] = sum(norm[c] * pesos_ahp[c] for c in crit)
municipios[["municipio", "puntaje_suma", "puntaje_ahp"]].sort_values("puntaje_ahp", ascending=False).round(3)

## 5. TOPSIS

En TOPSIS la mejor alternativa es la que está más cerca de la solución ideal, es decir lo mejor
en cada criterio, y más lejos de la peor. Se aplica sobre los criterios normalizados y ponderados.

In [ ]:
M = norm[crit].values * np.array([pesos_ahp[c] for c in crit])   # normalizada * peso
ideal      = M.max(axis=0)     # lo mejor de cada criterio
anti_ideal = M.min(axis=0)     # lo peor

d_ideal      = np.sqrt(((M - ideal) ** 2).sum(axis=1))
d_anti_ideal = np.sqrt(((M - anti_ideal) ** 2).sum(axis=1))
municipios["puntaje_topsis"] = d_anti_ideal / (d_ideal + d_anti_ideal)   # cercania (0 a 1)
municipios[["municipio", "puntaje_topsis"]].sort_values("puntaje_topsis", ascending=False).round(3)

## 6. Comparativa: mismos datos, distinto método

Los tres métodos, lado a lado, con el ranking de cada uno. Sobre los mismos datos y el mismo
mapa, el resultado cambia según el método.

Nota: la suma ponderada usó pesos puestos a mano (0.5 / 0.2 / 0.3) y AHP derivó los suyos de la
matriz de comparación (0.571 / 0.143 / 0.286). Los pesos son distintos a propósito: son dos formas
de fijarlos, no un error. Por eso los rankings pueden diferir un poco.

In [ ]:
tabla = municipios[["municipio", "puntaje_suma", "puntaje_ahp", "puntaje_topsis"]].copy()
for c in ["puntaje_suma", "puntaje_ahp", "puntaje_topsis"]:
    tabla[c.replace("puntaje", "rank")] = tabla[c].rank(ascending=False).astype(int)
tabla.sort_values("puntaje_ahp", ascending=False).round(3)

In [ ]:
tabla.set_index("municipio")[["puntaje_suma", "puntaje_ahp", "puntaje_topsis"]].plot(
    kind="bar", figsize=(9, 4))
plt.ylabel("Puntaje (0 a 1)"); plt.title("Idoneidad por metodo")
plt.tight_layout(); plt.show()

### Mapa de idoneidad (AHP)

Llevamos el puntaje AHP al mapa. Cada municipio va a su celda H3 y se colorea por su puntaje
(rojo = más idóneo).

In [ ]:
import branca.colormap as cm
colormap = cm.LinearColormap(["blue", "yellow", "red"],
                             vmin=municipios["puntaje_ahp"].min(),
                             vmax=municipios["puntaje_ahp"].max())
colormap.caption = "Idoneidad (AHP)"

m = folium.Map(location=[10.5, -73.0], zoom_start=7, tiles="CartoDB positron")
for _, r in municipios.iterrows():
    borde = h3.cell_to_boundary(r["h3_index"])
    folium.Polygon([[la, lo] for la, lo in borde], color="grey", weight=1,
                   fill=True, fill_color=colormap(r["puntaje_ahp"]), fill_opacity=0.75,
                   tooltip=f"{r['municipio']}: {r['puntaje_ahp']:.2f}").add_to(m)
colormap.add_to(m)
m

### Guardar el resultado

El resultado se guarda en el formato de trabajo del proyecto: `.h3.parquet` (celda H3 + puntajes).
Este archivo es el que pasa al Módulo 4.

In [ ]:
resultado = municipios[["h3_index", "municipio", "puntaje_suma", "puntaje_ahp",
                        "puntaje_topsis"]].merge(
    tabla[["municipio", "rank_suma", "rank_ahp", "rank_topsis"]], on="municipio")
resultado.to_parquet("resultado_idoneidad.h3.parquet", index=False)
print("guardado resultado_idoneidad.h3.parquet")
resultado.round(3)

## Actividad individual

Modifique la matriz de comparación AHP para que la radiación sea el criterio más importante y
observe cómo cambia el ranking.

Escriba su código en la celda siguiente. Más abajo hay una solución posible.

In [ ]:
# Escriba su codigo aqui (parta de la matriz A del paso 4)


<details><summary>Ver solución posible</summary>

```python
# radiacion mas importante que viento y que distancia
A2 = np.array([
    [1,   2,   1/3],   # viento
    [1/2, 1,   1/4],   # dist_via
    [3,   4,   1  ],   # radiacion (la mas importante)
], dtype=float)
pesos2 = dict(zip(crit, (A2 / A2.sum(0)).mean(1)))
print({c: round(w, 3) for c, w in pesos2.items()})

municipios["puntaje_ahp2"] = sum(norm[c] * pesos2[c] for c in crit)
municipios[["municipio", "puntaje_ahp", "puntaje_ahp2"]].sort_values(
    "puntaje_ahp2", ascending=False).round(3)
# Uribia sigue liderando; Maicao sube por encima de Riohacha.
```
</details>

## Cierre

Vimos tres formas de combinar criterios en un puntaje de idoneidad:

- **Suma ponderada**: pesos directos, simple y transparente.
- **AHP**: pesos derivados de comparaciones por pares, con chequeo de consistencia.
- **TOPSIS**: cercanía a la solución ideal.

Los tres usan los mismos datos y producen un ranking; según el método y los pesos, el resultado
cambia. Elegir el método y justificar los pesos es parte de la decisión.

El taller del Módulo 4 junta todo: datos crudos en distintos sistemas de coordenadas,
unificación, malla y uno de estos métodos para producir el mapa final.